# AUTO_01_Orders_Ingestion — Production

Production-ready incremental ingestion for the Orders dataset.

This notebook is designed to be executed by a Databricks/Lakeflow Job. A file-arrival trigger will start the Job when a new file arrives in the Orders S3 landing folder. The notebook itself does not create test files and does not contain hardcoded row-count assumptions.

**Source:** `s3://olist-retail-project/raw/orders/`  
**Target:** `workspace.bronze.orders`  
**Mode:** Incremental append using Auto Loader  
**Schema policy:** Strict; unexpected new columns fail the ingestion  
**Checkpoint:** `s3://olist-retail-project/_checkpoints/orders_ingestion/`


In [0]:
# ================================================================
# CELL 1 — AUTOMATED ORDERS INGESTION CONFIGURATION
# ================================================================
# Defines the S3 landing folder, Auto Loader checkpoint, and the
# existing Bronze target. No filename or row count is hardcoded.
# ================================================================

from pyspark.sql import functions as F

SOURCE_PATH = "s3://olist-retail-project/raw/orders/"

CHECKPOINT_PATH = (
    "s3://olist-retail-project/_checkpoints/orders_ingestion/"
)

BRONZE_TABLE = "workspace.bronze.orders"

print("Source     :", SOURCE_PATH)
print("Checkpoint :", CHECKPOINT_PATH)
print("Target     :", BRONZE_TABLE)


Source     : s3://olist-retail-project/raw/orders/
Checkpoint : s3://olist-retail-project/_checkpoints/orders_ingestion/
Target     : workspace.bronze.orders


In [0]:
# ================================================================
# CELL 2 — VERIFY EXISTING BRONZE TARGET
# ================================================================
# Confirms that the Bronze table already exists before ingestion.
# The current row count is reported for visibility only; it is not
# used as a hardcoded production expectation.
# ================================================================

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )

before_count = spark.table(BRONZE_TABLE).count()

print(f"Current Bronze Orders rows : {before_count:,}")
print(f"Bronze target verified      : {BRONZE_TABLE}")


Current Bronze Orders rows : 99,446
Bronze target verified      : workspace.bronze.orders


In [0]:
# ================================================================
# CELL 3 — USE EXISTING BRONZE SCHEMA AS THE CONTRACT
# ================================================================
# Reads the current Bronze Orders schema dynamically so the
# ingestion remains aligned with the existing Bronze architecture.
# ================================================================

bronze_schema = spark.table(BRONZE_TABLE).schema

print("Bronze schema contract:")
for field in bronze_schema:
    print(
        f"  {field.name:<40} {field.dataType.simpleString()}"
    )


Bronze schema contract:
  order_id                                 string
  customer_id                              string
  order_status                             string
  order_purchase_timestamp                 string
  order_approved_at                        string
  order_delivered_carrier_date             string
  order_delivered_customer_date            string
  order_estimated_delivery_date            string


In [0]:
# ================================================================
# CELL 4 — CREATE AUTO LOADER STREAM
# ================================================================
# Auto Loader discovers newly arriving CSV files in the Orders
# landing folder. Existing historical files are excluded because
# they are already represented in the existing Bronze table.
# The existing Bronze schema is enforced as the input contract.
# ================================================================

orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(bronze_schema)
        .option(
            "cloudFiles.schemaEvolutionMode",
            "failOnNewColumns"
        )
        .option(
            "cloudFiles.includeExistingFiles",
            "false"
        )
        .load(SOURCE_PATH)
)

print("PASS — Auto Loader stream configured for new Orders files.")


PASS — Auto Loader stream configured for new Orders files.


In [0]:
# ================================================================
# CELL 5 — INCREMENTAL WRITE TO EXISTING BRONZE TABLE
# ================================================================
# Appends newly discovered records to the existing Bronze table.
# The checkpoint stores Auto Loader progress so a processed file
# is not processed again on a later Job run.
#
# availableNow=True is intentional: the Job processes all files
# currently available and then terminates. The next file-arrival
# trigger starts a fresh Job run using the same checkpoint.
# ================================================================

query = (
    orders_stream
        .writeStream
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print(
    "PASS — Orders incremental ingestion completed successfully."
)


PASS — Orders incremental ingestion completed successfully.


In [0]:
# ================================================================
# CELL 6 — POST-INGESTION VALIDATION
# ================================================================
# Reports the current Bronze row count after ingestion. No fixed
# expected count is used because incoming file sizes can vary.
# ================================================================

after_count = spark.table(BRONZE_TABLE).count()

print(f"Current Bronze Orders rows : {after_count:,}")
print(f"Bronze target              : {BRONZE_TABLE}")

print(
    "PASS — Bronze Orders table is available after ingestion."
)


Current Bronze Orders rows : 99,446
Bronze target              : workspace.bronze.orders
PASS — Bronze Orders table is available after ingestion.


In [0]:
# ================================================================
# CELL 7 — POST-INGESTION SCHEMA VALIDATION
# ================================================================
# Confirms that the target Bronze schema remains unchanged after
# the ingestion run.
# ================================================================

current_schema = spark.table(BRONZE_TABLE).schema

if current_schema != bronze_schema:
    raise ValueError(
        "Bronze Orders schema changed unexpectedly."
    )

print("PASS — Bronze Orders schema remains unchanged.")

print("Current Bronze columns:")

for field in current_schema:
    print(
        f"  {field.name:<40} {field.dataType.simpleString()}"
    )


PASS — Bronze Orders schema remains unchanged.
Current Bronze columns:
  order_id                                 string
  customer_id                              string
  order_status                             string
  order_purchase_timestamp                 string
  order_approved_at                        string
  order_delivered_carrier_date             string
  order_delivered_customer_date            string
  order_estimated_delivery_date            string


In [0]:
# ================================================================
# CELL 8 — BRONZE BUSINESS-KEY VALIDATION
# ================================================================
# Checks that non-null order_id values remain unique. This is a
# data-quality control rather than a hardcoded row-count check.
# ================================================================

duplicate_orders = (
    spark.table(BRONZE_TABLE)
        .filter(F.col("order_id").isNotNull())
        .groupBy("order_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    f"Duplicate non-null order_id values : {duplicate_orders:,}"
)

if duplicate_orders > 0:
    raise ValueError(
        "Bronze Orders quality check failed: "
        "duplicate order_id values detected."
    )

print(
    "PASS — Bronze Orders order_id uniqueness check passed."
)


Duplicate non-null order_id values : 0
PASS — Bronze Orders order_id uniqueness check passed.


In [0]:
# ================================================================
# CELL 9 — FINAL INGESTION STATUS
# ================================================================
# Produces a concise status message for Databricks Job logs.
# ================================================================

print("=" * 70)
print("ORDERS AUTOMATED INGESTION — SUCCESS")
print("=" * 70)
print(f"Source       : {SOURCE_PATH}")
print(f"Target       : {BRONZE_TABLE}")
print(f"Current rows : {after_count:,}")
print("Mode         : Incremental append")
print("File handling: Auto Loader")
print("Schema mode  : Strict")
print("=" * 70)


ORDERS AUTOMATED INGESTION — SUCCESS
Source       : s3://olist-retail-project/raw/orders/
Target       : workspace.bronze.orders
Current rows : 99,446
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict
